# Factor Diagnostics — Why Is Sharpe Negative?

The pipeline backtest returned **Sharpe = -0.56** and **Total Return = -5.94%**.
This notebook diagnoses the problem by examining each factor individually:

1. **Per-factor IC and IR** — which factors predict returns, and in which direction?
2. **Cumulative long-short returns** — does going long top-ranked stocks actually work?
3. **Factor direction check** — are we ranking the wrong way?
4. **Combination analysis** — does equal-weight combination help or hurt?
5. **AlphaConfig demo** — run the pipeline with corrected config

Reference: Quant Notes Entry 1 (weak IR by design) & Entry 2 (IC sign + anti-correlation).

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from data.loader.data_loader import stock_load_process
from data.universe import get_universe
from constants import DATE_COL, TICKER_COL, VALUE_COL, TRADING_DAYS_PER_YEAR

## 1. Load Data

In [2]:
UNIVERSE = get_universe("US_LARGE_CAP_50")
START_DATE = "2023-01-01"
END_DATE = "2026-02-28"

ohlcv = (
    stock_load_process(tickers=UNIVERSE, start_date=START_DATE, end_date=END_DATE)
    .filter(pl.col("volume") != 0)
    .collect()
)
print(f"OHLCV: {ohlcv.shape[0]:,} rows, {ohlcv.select('ticker').n_unique()} tickers")

Loading from cache: /mnt/blackdisk/quant_data/polygon_data/processed/us_stocks_sip/day_aggs_v1/cache_c5e11349e9e04e8bdef2634ae48ac375.parquet
Cache loaded: 40,301 rows, 2.58 MB
OHLCV: 40,093 rows, 52 tickers


## 2. Compute Each Factor + Daily Returns

In [3]:
from portfolio.pipeline import compute_daily_returns, compute_next_day_returns
from portfolio.factors import get_factor_fn, list_factors
from portfolio.alpha_config import FactorConfig

daily_returns = compute_daily_returns(ohlcv)
next_day_returns = compute_next_day_returns(daily_returns)

print(f"Available factors: {list_factors()}")
print(f"Daily returns: {daily_returns.shape[0]:,} rows")
print(f"Next-day returns: {next_day_returns.shape[0]:,} rows")

Available factors: ['bbiboll', 'momentum', 'vol_ratio']
Daily returns: 40,041 rows
Next-day returns: 39,989 rows


In [4]:
# Compute each factor individually
FACTOR_NAMES = ["bbiboll", "vol_ratio", "momentum"]

factors = {}
for name in FACTOR_NAMES:
    fn = get_factor_fn(name)
    factors[name] = fn(ohlcv, factor_config=FactorConfig())
    n_rows = factors[name].shape[0]
    n_tickers = factors[name].select(TICKER_COL).n_unique()
    print(f"{name}: {n_rows:,} rows, {n_tickers} tickers")

bbiboll: 38,377 rows, 52 tickers
vol_ratio: 39,053 rows, 52 tickers
momentum: 39,053 rows, 52 tickers


## 3. Per-Factor IC Analysis

IC = Spearman rank correlation between factor value and next-day return, computed daily.

In [5]:
def compute_daily_ic(factor_df: pl.DataFrame, ndr: pl.DataFrame) -> pl.DataFrame:
    """Compute daily IC (rank correlation) between factor and next-day return."""
    merged = factor_df.join(ndr, on=[DATE_COL, TICKER_COL], how="inner")
    ic_series = (
        merged.group_by(DATE_COL)
        .agg(
            pl.corr(pl.col(VALUE_COL).rank(), pl.col("next_day_return").rank())
            .alias("ic")
        )
        .sort(DATE_COL)
        .filter(pl.col("ic").is_not_null() & pl.col("ic").is_finite())
    )
    return ic_series


ic_results = {}
for name, factor_df in factors.items():
    ic_df = compute_daily_ic(factor_df, next_day_returns)
    ic_arr = ic_df["ic"].to_numpy()
    mean_ic = float(np.mean(ic_arr))
    std_ic = float(np.std(ic_arr, ddof=1))
    ir = mean_ic / std_ic if std_ic > 1e-10 else 0.0
    ic_results[name] = {
        "ic_series": ic_df,
        "mean_ic": mean_ic,
        "std_ic": std_ic,
        "ir": ir,
        "t_stat": mean_ic / (std_ic / np.sqrt(len(ic_arr))) if std_ic > 1e-10 else 0.0,
        "n_days": len(ic_arr),
    }

# Summary table
print(f"{'Factor':<12} {'Mean IC':>10} {'Std IC':>10} {'IR':>10} {'t-stat':>10} {'N days':>8}")
print("-" * 64)
for name, r in ic_results.items():
    print(f"{name:<12} {r['mean_ic']:>10.4f} {r['std_ic']:>10.4f} {r['ir']:>10.4f} {r['t_stat']:>10.2f} {r['n_days']:>8}")

Factor          Mean IC     Std IC         IR     t-stat   N days
----------------------------------------------------------------
bbiboll          0.0056     0.2234     0.0251       0.69      752
vol_ratio        0.0006     0.1704     0.0033       0.09      765
momentum        -0.0034     0.2498    -0.0137      -0.38      765


In [6]:
# Plot cumulative IC for each factor
fig = make_subplots(rows=len(FACTOR_NAMES), cols=1, shared_xaxes=True,
                    subplot_titles=[f"{n} — Cumulative IC (mean={ic_results[n]['mean_ic']:.4f}, IR={ic_results[n]['ir']:.3f})"
                                   for n in FACTOR_NAMES])

colors = px.colors.qualitative.Set2
for i, name in enumerate(FACTOR_NAMES):
    ic_df = ic_results[name]["ic_series"]
    dates = ic_df[DATE_COL].to_list()
    cum_ic = np.cumsum(ic_df["ic"].to_numpy())
    fig.add_trace(
        go.Scatter(x=dates, y=cum_ic, name=name,
                   line=dict(color=colors[i % len(colors)])),
        row=i + 1, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=i + 1, col=1)

fig.update_layout(height=250 * len(FACTOR_NAMES), title="Cumulative IC by Factor",
                  showlegend=False)
fig.show()

## 4. Long-Short Portfolio Returns by Factor

For each factor: go long top-10, short bottom-10 (equal weight), daily rebalance.
This shows whether the factor's *direction* is correct.

In [6]:
from risk.position_sizing import size_equal_weight
from portfolio.pipeline import compute_portfolio_return

N_LONG = 10
N_SHORT = 10

ls_returns = {}
for name, factor_df in factors.items():
    weights = size_equal_weight(factor_df, n_long=N_LONG, n_short=N_SHORT)
    port_ret = compute_portfolio_return(weights, next_day_returns)
    ret_arr = port_ret["port_return"].to_numpy()
    cum_ret = np.cumprod(1 + ret_arr)
    mu = float(np.mean(ret_arr))
    sigma = float(np.std(ret_arr, ddof=1))
    sharpe = mu / sigma * np.sqrt(TRADING_DAYS_PER_YEAR) if sigma > 1e-10 else 0.0
    total_ret = float(cum_ret[-1] - 1) if len(cum_ret) > 0 else 0.0
    ls_returns[name] = {
        "dates": port_ret[DATE_COL].to_list(),
        "cum_return": cum_ret,
        "sharpe": sharpe,
        "total_return": total_ret,
        "annual_return": mu * TRADING_DAYS_PER_YEAR,
    }

print(f"{'Factor':<12} {'Sharpe':>10} {'Total Ret':>12} {'Annual Ret':>12}")
print("-" * 50)
for name, r in ls_returns.items():
    print(f"{name:<12} {r['sharpe']:>10.3f} {r['total_return']:>11.2%} {r['annual_return']:>11.2%}")

Factor           Sharpe    Total Ret   Annual Ret
--------------------------------------------------
bbiboll           0.814      21.43%       6.86%
vol_ratio        -0.137      -3.68%      -0.98%
momentum          0.080       0.88%       0.84%


In [9]:
# Plot cumulative L/S returns
fig = go.Figure()
for i, (name, r) in enumerate(ls_returns.items()):
    fig.add_trace(go.Scatter(
        x=r["dates"], y=r["cum_return"],
        name=f"{name} (Sharpe={r['sharpe']:.2f})",
        line=dict(color=colors[i % len(colors)])
    ))

fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Cumulative Long-Short Returns by Factor (Top 10 / Bottom 10)",
    yaxis_title="Cumulative Return (1.0 = flat)",
    height=500,
)
fig.show()

## 5. Factor Direction Check

If a factor has **negative IC**, it means high factor value → low return.
The pipeline ranks by factor value (high = long), so a negative-IC factor
is being traded *backwards*.

**Fix:** negate the factor (multiply by -1) before ranking.

Let's test what happens when we flip the sign of negative-IC factors.

In [7]:
# Identify which factors need flipping
flip_factors = [name for name, r in ic_results.items() if r["mean_ic"] < 0]
keep_factors = [name for name, r in ic_results.items() if r["mean_ic"] >= 0]

print(f"Factors with NEGATIVE IC (need sign flip): {flip_factors}")
print(f"Factors with POSITIVE IC (keep as-is):     {keep_factors}")

# Flip the sign and recompute L/S returns
ls_returns_flipped = {}
for name, factor_df in factors.items():
    if name in flip_factors:
        factor_flipped = factor_df.with_columns(-pl.col(VALUE_COL))
        label = f"{name} (flipped)"
    else:
        factor_flipped = factor_df
        label = name

    weights = size_equal_weight(factor_flipped, n_long=N_LONG, n_short=N_SHORT)
    port_ret = compute_portfolio_return(weights, next_day_returns)
    ret_arr = port_ret["port_return"].to_numpy()
    cum_ret = np.cumprod(1 + ret_arr)
    mu = float(np.mean(ret_arr))
    sigma = float(np.std(ret_arr, ddof=1))
    sharpe = mu / sigma * np.sqrt(TRADING_DAYS_PER_YEAR) if sigma > 1e-10 else 0.0
    total_ret = float(cum_ret[-1] - 1) if len(cum_ret) > 0 else 0.0
    ls_returns_flipped[label] = {
        "dates": port_ret[DATE_COL].to_list(),
        "cum_return": cum_ret,
        "sharpe": sharpe,
        "total_return": total_ret,
    }

print(f"\n{'Factor':<20} {'Sharpe':>10} {'Total Ret':>12}")
print("-" * 45)
for name, r in ls_returns_flipped.items():
    print(f"{name:<20} {r['sharpe']:>10.3f} {r['total_return']:>11.2%}")

Factors with NEGATIVE IC (need sign flip): ['momentum']
Factors with POSITIVE IC (keep as-is):     ['bbiboll', 'vol_ratio']

Factor                   Sharpe    Total Ret
---------------------------------------------
bbiboll                   0.814      21.43%
vol_ratio                -0.137      -3.68%
momentum (flipped)       -0.080      -4.14%


In [11]:
# Plot flipped vs original
fig = go.Figure()
for i, (name, r) in enumerate(ls_returns_flipped.items()):
    fig.add_trace(go.Scatter(
        x=r["dates"], y=r["cum_return"],
        name=f"{name} (Sharpe={r['sharpe']:.2f})",
        line=dict(color=colors[i % len(colors)])
    ))

fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Long-Short Returns — After Flipping Negative-IC Factors",
    yaxis_title="Cumulative Return",
    height=500,
)
fig.show()

## 6. IC Correlation Matrix

Check factor orthogonality. Recall from Quant Notes Entry 2:
- Same IC sign + anti-correlation → great diversification
- Opposite IC sign + anti-correlation → cancellation

In [8]:
# Build IC matrix: align all IC series on common dates
ic_wide = None
for name in FACTOR_NAMES:
    ic_df = ic_results[name]["ic_series"].rename({"ic": f"ic_{name}"})
    if ic_wide is None:
        ic_wide = ic_df
    else:
        ic_wide = ic_wide.join(ic_df, on=DATE_COL, how="inner")

ic_cols = [f"ic_{n}" for n in FACTOR_NAMES]
ic_np = ic_wide.select(ic_cols).to_numpy()
corr_matrix = np.corrcoef(ic_np.T)

print("IC Correlation Matrix:")
print(f"{'':>12}", end="")
for n in FACTOR_NAMES:
    print(f"{n:>12}", end="")
print()
for i, n in enumerate(FACTOR_NAMES):
    print(f"{n:>12}", end="")
    for j in range(len(FACTOR_NAMES)):
        print(f"{corr_matrix[i, j]:>12.3f}", end="")
    print()

IC Correlation Matrix:
                 bbiboll   vol_ratio    momentum
     bbiboll       1.000       0.034       0.581
   vol_ratio       0.034       1.000      -0.039
    momentum       0.581      -0.039       1.000


In [13]:
# Heatmap
fig = px.imshow(
    corr_matrix,
    x=FACTOR_NAMES, y=FACTOR_NAMES,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto=".3f",
    title="IC Correlation Matrix",
)
fig.update_layout(height=400, width=500)
fig.show()

## 7. Pipeline Comparison with AlphaConfig

Run the full pipeline with different configs to compare:
- Default (original, Sharpe ≈ -0.56)
- With factor direction correction
- Different combination methods

In [9]:
from portfolio.alpha_config import AlphaConfig, FactorConfig
from portfolio.pipeline import run_alpha_pipeline

configs = {
    "Default (bbiboll + vol_ratio)": AlphaConfig(
        factor_names=["bbiboll", "vol_ratio"],
        combination_method="equal_weight",
        sizing_method="Signal-Weighted",
        rebal_every_n=5,
        name="Default",
    ),
    "Single: bbiboll only": AlphaConfig(
        factor_names=["bbiboll"],
        sizing_method="Signal-Weighted",
        rebal_every_n=5,
        name="BBIBOLL-only",
    ),
    "Single: vol_ratio only": AlphaConfig(
        factor_names=["vol_ratio"],
        sizing_method="Signal-Weighted",
        rebal_every_n=5,
        name="VolRatio-only",
    ),
    "Single: momentum only": AlphaConfig(
        factor_names=["momentum"],
        sizing_method="Signal-Weighted",
        rebal_every_n=5,
        name="Momentum-only",
    ),
    "All 3 factors": AlphaConfig(
        factor_names=["bbiboll", "vol_ratio", "momentum"],
        combination_method="equal_weight",
        sizing_method="Signal-Weighted",
        rebal_every_n=5,
        name="All3-EW",
    ),
    "Equal-Weight sizing": AlphaConfig(
        factor_names=["bbiboll", "vol_ratio"],
        combination_method="equal_weight",
        sizing_method="Equal-Weight",
        rebal_every_n=5,
        name="EW-sizing",
    ),
}

results = {}
for label, cfg in configs.items():
    r = run_alpha_pipeline(ohlcv, config=cfg)
    results[label] = r
    print(f"{label:<35} Sharpe={r['sharpe']:>7.3f}  Return={r['annual_return']:>7.2%}  Vol={r['annual_vol']:>7.2%}")

Default (bbiboll + vol_ratio)       Sharpe= -0.820  Return=-12.75%  Vol= 15.54%
Single: bbiboll only                Sharpe= -0.077  Return= -1.33%  Vol= 17.16%
Single: vol_ratio only              Sharpe= -0.323  Return= -4.69%  Vol= 14.50%
Single: momentum only               Sharpe=  0.151  Return=  3.17%  Vol= 20.96%
All 3 factors                       Sharpe= -0.544  Return=-10.52%  Vol= 19.32%
Equal-Weight sizing                 Sharpe= -0.147  Return= -1.12%  Vol=  7.64%


In [15]:
# Plot cumulative equity curves for all configs
fig = go.Figure()
for i, (label, r) in enumerate(results.items()):
    cum_ret = np.cumprod(1 + r["returns_array"])
    dates = r["portfolio_returns"][DATE_COL].to_list()
    fig.add_trace(go.Scatter(
        x=dates, y=cum_ret,
        name=f"{label} (S={r['sharpe']:.2f})",
        line=dict(color=colors[i % len(colors)])
    ))

fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Pipeline Equity Curves — AlphaConfig Comparison",
    yaxis_title="Cumulative Return",
    height=600,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 8. Diagnosis Summary

After running the cells above, fill in these findings:

| Factor | Mean IC | IR | IC Sign | Direction Correct? |
|--------|---------|-----|---------|--------------------|
| bbiboll | 0.0056 | 0.0251 | + | yes |
| vol_ratio | 0.0006 | 0.0033 | + | yes |
| momentum | -0.0034 | -0.0137 | - | no |

### Next Steps

1. **Flip negative-IC factors** — negate factor values before ranking
2. **Drop factors with |IR| < 0.03** — too noisy to be useful
3. **Try IC-weighted combination** — weight factors by their predictive power
4. **Add new factors** — 12-1 month momentum, short-term reversal
5. **Record findings** — add Entry 4 to Quant Notes (docs/quant_lab.tex)